In [ ]:
!pip install -q datasets

In [3]:
from datasets import load_dataset

# Stream the dataset to avoid downloading the full ~300MB split,
# shuffle with a buffer, and materialize the first 10k rows as a list.
stream = load_dataset("nvidia/Nemotron-PII", split="train", streaming=True)
stream = stream.shuffle(seed=42, buffer_size=10_000).take(10_000)
ds = list(stream)
print("num rows:", len(ds))
print("columns:", list(ds[0].keys()))

num rows: 10000
columns: ['uid', 'domain', 'document_type', 'document_description', 'document_format', 'locale', 'text', 'spans', 'text_tagged']


In [4]:
import ast

row = ds[0]
print("keys:", list(row.keys()))
print("text[:300]:", row["text"][:300])
print()
print("spans (raw type):", type(row["spans"]).__name__)
spans = ast.literal_eval(row["spans"])
print("first 5 spans:")
for sp in spans[:5]:
    print(" ", sp)

keys: ['uid', 'domain', 'document_type', 'document_description', 'document_format', 'locale', 'text', 'spans', 'text_tagged']
text[:300]: **Patent Application**

**Title: Secure Legal Document Management System**

**Abstract:**

This patent application, submitted by LexGuard Legal under the unique id: 9876543210, discloses a revolutionary system for managing legal documents securely. The system, developed by user name: tonipemberton, 

spans (raw type): str
first 5 spans:
  {'start': 129, 'end': 143, 'text': 'LexGuard Legal', 'label': 'company_name'}
  {'start': 165, 'end': 175, 'text': '9876543210', 'label': 'unique_id'}
  {'start': 285, 'end': 298, 'text': 'tonipemberton', 'label': 'user_name'}
  {'start': 498, 'end': 508, 'text': '15.08.2028', 'label': 'date'}
  {'start': 1829, 'end': 1865, 'text': 'https://uspto.gov/patent/application', 'label': 'url'}


In [5]:
from collections import Counter

label_counts = Counter()
for row in ds:
    try:
        spans = ast.literal_eval(row["spans"])
    except Exception:
        continue
    for sp in spans:
        lab = sp.get("label")
        if lab:
            label_counts[lab] += 1

print(f"unique labels: {len(label_counts)}")
for lab, n in label_counts.most_common():
    print(f"  {n:>7}  {lab}")

unique labels: 54
     8256  company_name
     7427  date
     6289  first_name
     4741  url
     4722  last_name
     3962  email
     3342  occupation
     2525  phone_number
     2227  time
     2198  country
     2124  state
     1960  street_address
     1724  city
     1587  customer_id
     1345  employee_id
     1311  county
     1276  certificate_license_number
     1234  user_name
     1183  date_time
     1183  biometric_identifier
     1155  education_level
     1071  date_of_birth
     1064  account_number
      961  coordinate
      934  credit_debit_card
      925  ipv4
      916  vehicle_identifier
      883  postcode
      874  password
      836  employment_status
      789  license_plate
      781  medical_record_number
      709  fax_number
      681  bank_routing_number
      673  race_ethnicity
      666  health_plan_beneficiary_number
      660  pin
      623  ssn
      601  mac_address
      583  http_cookie
      581  swift_bic
      566  device_identifier
  

In [7]:
import re

pat = re.compile(r"\b(P\.?\s*O\.?\s*Box|PO\s*Box|Post Office Box)\b", re.IGNORECASE)

hits = []
for row in ds:
    m = pat.search(row["text"] or "")
    if not m:
        continue
    try:
        spans = ast.literal_eval(row["spans"])
    except Exception:
        continue
    hits.append((row["text"], spans, m.start()))
    if len(hits) >= 20:
        break

print(f"rows with PO Box pattern (first {len(hits)}):")
for text, spans, pos in hits:
    snippet = text[max(0, pos - 20):pos + 80].replace("\n", " ")
    print(f"\n--- ctx: ...{snippet}...")
    for sp in spans:
        if abs(sp.get("start", -999) - pos) < 100:
            print(f"    [{sp['label']}] {sp['text']!r}  (start={sp['start']})")

rows with PO Box pattern (first 0):


In [8]:
def labels_near(pattern: str, max_rows: int = 5000):
    """Return Counter of labels whose span starts within 80 chars of `pattern` matches."""
    pat = re.compile(pattern, re.IGNORECASE)
    counts = Counter()
    examined = 0
    for row in ds:
        if examined >= max_rows:
            break
        m = pat.search(row["text"] or "")
        if not m:
            continue
        examined += 1
        try:
            spans = ast.literal_eval(row["spans"])
        except Exception:
            continue
        for sp in spans:
            if abs(sp.get("start", -999) - m.start()) < 80:
                counts[sp["label"]] += 1
    return examined, counts

n, c = labels_near(r"\bP\.?\s*O\.?\s*Box\b")
print(f"PO Box matches inspected: {n}")
print(f"nearby labels: {dict(c)}")

PO Box matches inspected: 0
nearby labels: {}
